# Lightweight Self-Supervised Text Representation Learning
## Google Colab / Kaggle Three-Variant Training & Geometry Benchmark

**Project:** A Lightweight Self-Supervised Framework for Text Representation Learning Using Masked Latent Prediction.

This executable notebook investigates the core research question under a fixed small English text encoder ($L=128, D=256$, 4 layers, 4 attention heads) by comparing:
- **Variant C (MLM Control):** True token prediction via vocabulary cross-entropy (`configs/mlm_c.json`).
- **Variant A (Masked Latent Anchor):** Final teacher layer target, short contiguous spans (~15%), 2-layer attention predictor, MSE loss (`configs/latent_a.json`).
- **Variant B (Masked Latent Anchor):** Average of normalized top-2 teacher layers, independent tokens (~15%), linear predictor, Smooth L1 loss (`configs/latent_b.json`).

### Experimental Workflow:
1. **Pin Commit:** Checks out explicit verified commit (`8120215` or later).
2. **Environment & Tests:** Installs verified dependencies and runs the complete test suite.
3. **Durable Artifact Storage:** Mounts Google Drive or sets up Kaggle output directory for persistent storage across runtime resets.
4. **Real Data Ingestion:** Loads real WikiText pilot data via `src.data.get_dataset_loaders` strictly enforcing `docs/INTERFACES.md`.
5. **Three-Variant Training:** Executes `Trainer` and `run_batches` for Variants C, A, and B, using `SmallEncoder`.
6. **Loss Curves:** Plots and compares real measured training losses.
7. **Restart Recovery:** Tests durable checkpoint reload after memory clearing and simulates a runtime restart.
8. **Representation Geometry:** Evaluates centered effective rank (RankMe), dimension variance, and cosine distributions on held-out validation representations.
9. **Downstream Transfer:** Trains and evaluates frozen linear probes on AG News across all three variants.

> **Security Notice:** Never embed GitHub access tokens in notebooks or commit credentials to git. Use Colab Secrets (`userdata.get('GH_TOKEN')`) or enter credentials interactively when cloning private repositories.

### 1. Hardware & Environment Inspection

In [ ]:
!nvidia-smi

import os
import sys
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__}")
print(f"Selected Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name:        {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:      {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

### 2. Git Setup & Checkout to Pinned Commit

In [ ]:
PINNED_COMMIT = "673c14e"  # Pinned integration commit (8120215 or later)

if not os.path.exists("src"): 
    print("Cloning repository...")
    !git clone https://github.com/dakshadev/7th-sem.git .

!git fetch origin person2
!git checkout {PINNED_COMMIT}
!git log -n 1 --oneline

### 3. Dependency Installation & Complete Test Suite Verification
Verifies all unit and integration tests across both Person 1 and Person 2 modules.

In [ ]:
!pip install -q -r requirements.txt
!python -m pytest -q

### 4. Durable Storage Path Setup (Google Drive / Kaggle Output)
Configures a persistent directory outside ephemeral runtime memory so that checkpoints, metrics, and plots survive runtime resets.

In [ ]:
from pathlib import Path

# Determine runtime environment
if os.path.exists("/content"):  # Google Colab
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        DURABLE_ROOT = Path("/content/drive/MyDrive/7thSem_artifacts")
    except Exception as e:
        print(f"Note: Drive mount skipped ({e}). Using local fallback.")
        DURABLE_ROOT = Path("artifacts/durable")
elif os.path.exists("/kaggle/working"):  # Kaggle
    DURABLE_ROOT = Path("/kaggle/working/artifacts")
else:
    DURABLE_ROOT = Path("artifacts/durable")

DURABLE_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Durable artifacts directory: {DURABLE_ROOT.resolve()}")

### 5. Real Dataset Preparation (`src.data.get_dataset_loaders`)
Loads real WikiText pilot data (`wikitext-2-raw-v1`, ~2M tokens) and validates the contract batch tensors.

In [ ]:
from src.data import get_dataset_loaders, get_bert_tokenizer, count_tokens_in_batch

tokenizer = get_bert_tokenizer("bert-base-uncased")
train_loader, val_loader, data_manifest = get_dataset_loaders(
    dataset_config="wikitext-2-raw-v1",
    batch_size=8,
    max_length=128,
    mask_ratio=0.15,
    mask_mode="independent",
    seed=42,
    max_train_samples=600,
    max_val_samples=150,
    tokenizer=tokenizer,
)

print("Corpus Manifest:", data_manifest)
sample_batch = next(iter(train_loader))
print("Contract Tensors Verified:")
for k, v in sample_batch.items():
    print(f"  {k:18s} shape={list(v.shape)} dtype={v.dtype}")
print("Sample batch tokens:", count_tokens_in_batch(sample_batch))

### 6. Variant C Pretraining: Masked Language Modeling Control (`configs/mlm_c.json`)
Standard MLM baseline: cross-entropy loss against true token identities at independent masked positions.

In [ ]:
from src.config import load_config
from src.train import Trainer, run_batches

config_c = load_config("configs/mlm_c.json")
trainer_c = Trainer(config_c, device=device)
out_dir_c = DURABLE_ROOT / "mlm_c"

print(f"Training Variant C (MLM) on {device}...")
metrics_c = run_batches(
    trainer=trainer_c,
    batches=train_loader,
    output_dir=out_dir_c,
    segment_updates=50,
    validation_batches=val_loader,
    data_manifest=data_manifest,
)
trainer_c.export_student(out_dir_c / "student.pt")
print(f"Variant C complete! Exported student to {out_dir_c / 'student.pt'}")

### 7. Variant A Pretraining: Masked Latent Anchor (`configs/latent_a.json`)
Masked latent prediction recipe A:
- Final teacher layer representation target.
- Short contiguous spans masking (~15%).
- 2-layer attention predictor.
- MSE regression loss.

In [ ]:
# Variant A uses span masking (~15% eligible tokens)
train_loader_span, val_loader_span, manifest_span = get_dataset_loaders(
    dataset_config="wikitext-2-raw-v1",
    batch_size=8,
    max_length=128,
    mask_ratio=0.15,
    mask_mode="span",
    seed=42,
    max_train_samples=600,
    max_val_samples=150,
    tokenizer=tokenizer,
)

config_a = load_config("configs/latent_a.json")
trainer_a = Trainer(config_a, device=device)
out_dir_a = DURABLE_ROOT / "latent_a"

print(f"Training Variant A (Latent Anchor A) on {device}...")
metrics_a = run_batches(
    trainer=trainer_a,
    batches=train_loader_span,
    output_dir=out_dir_a,
    segment_updates=50,
    validation_batches=val_loader_span,
    data_manifest=manifest_span,
)
trainer_a.export_student(out_dir_a / "student.pt")
print(f"Variant A complete! Exported student to {out_dir_a / 'student.pt'}")

### 8. Variant B Pretraining: Masked Latent Anchor (`configs/latent_b.json`)
Masked latent prediction recipe B:
- Average of normalized top-2 teacher layers.
- Independent tokens masking (~15%).
- Linear predictor.
- Smooth L1 regression loss.

In [ ]:
config_b = load_config("configs/latent_b.json")
trainer_b = Trainer(config_b, device=device)
out_dir_b = DURABLE_ROOT / "latent_b"

print(f"Training Variant B (Latent Anchor B) on {device}...")
metrics_b = run_batches(
    trainer=trainer_b,
    batches=train_loader,
    output_dir=out_dir_b,
    segment_updates=50,
    validation_batches=val_loader,
    data_manifest=data_manifest,
)
trainer_b.export_student(out_dir_b / "student.pt")
print(f"Variant B complete! Exported student to {out_dir_b / 'student.pt'}")

### 9. Pretraining Loss Curves Comparison
> **Caution (docs/EXPERIMENT_PROTOCOL.md):** MLM cross-entropy, Latent MSE, and Smooth L1 measure different mathematical loss scales and must not be compared directly by magnitude. We plot them side-by-side to track optimization convergence within each objective.

In [ ]:
import json
import matplotlib.pyplot as plt

def read_losses(metrics_path):
    steps, losses = [], []
    if not metrics_path.exists():
        return steps, losses
    with open(metrics_path, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            if "loss" in row and row["loss"] is not None:
                steps.append(row["updates"])
                losses.append(row["loss"])
    return steps, losses

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
variants = [
    ("Variant C (MLM Control)", out_dir_c / "metrics.jsonl", "#1f77b4", axes[0]),
    ("Variant A (Latent MSE)", out_dir_a / "metrics.jsonl", "#ff7f0e", axes[1]),
    ("Variant B (Latent Smooth L1)", out_dir_b / "metrics.jsonl", "#2ca02c", axes[2]),
]

for title, path, color, ax in variants:
    steps, loss_vals = read_losses(path)
    ax.plot(steps, loss_vals, color=color, linewidth=2, label=title)
    ax.set_title(title, fontweight="bold", fontsize=11)
    ax.set_xlabel("Updates", fontsize=10)
    ax.set_ylabel("Objective Loss", fontsize=10)
    ax.grid(True, linestyle=":", alpha=0.6)
    ax.legend()

plt.suptitle("Measured Optimization Trajectories: Matched Pilot Segments", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(DURABLE_ROOT / "pilot_loss_trajectories.png", dpi=200)
plt.show()

### 10. Durable Checkpoint Recovery Test (Simulating Runtime Restart)
Verifies that all three variant checkpoints can be cleanly reloaded from persistent storage into fresh model instances, with frozen evaluation mode and intact configuration.

In [ ]:
import gc
from src.model import SmallEncoder
from src.evaluate import load_student_checkpoint

# 1. Wipe in-memory trainer instances to simulate runtime termination
del trainer_c, trainer_a, trainer_b
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 2. Reload exported student weights for each variant from durable storage
reloaded_students = {}
for var_name, var_dir in [("C", out_dir_c), ("A", out_dir_a), ("B", out_dir_b)]:
    student_path = var_dir / "student.pt"
    checkpoint_path = var_dir / "checkpoint.pt"
    assert student_path.exists(), f"Missing durable student export for {var_name}"
    assert checkpoint_path.exists(), f"Missing durable training checkpoint for {var_name}"
    
    # Load config and re-instantiate fresh SmallEncoder
    cfg = load_config(f"configs/{'mlm_c' if var_name == 'C' else 'latent_' + var_name.lower()}.json")
    model_instance = SmallEncoder(cfg.model).to(device)
    frozen_student, model_cfg = load_student_checkpoint(student_path, model_instance, return_config=True)
    
    assert not frozen_student.training, f"Model {var_name} is not in eval mode!"
    assert all(not p.requires_grad for p in frozen_student.parameters()), f"Model {var_name} parameters not frozen!"
    reloaded_students[var_name] = frozen_student
    print(f"Variant {var_name}: successfully reloaded and verified from {student_path}!")

print("\nAll 3 variants verified across simulated runtime restart!")

### 11. Representation Geometry Diagnostics on Held-Out Validation Split
Measures geometric collapse vs representation diversity across all three recipes on uncorrupted held-out text:
- **RankMe:** Centered effective rank using entropy of normalized singular values.
- **Covariance Rank:** Effective rank using entropy of normalized covariance eigenvalues.
- **Dimension Variance:** Mean sample variance and dead dimension percentage.
- **Pairwise Cosine:** Mean and standard deviation of centered pairwise cosine similarities.

In [ ]:
from src.evaluate import masked_mean_pooling, compute_representation_geometry

special_ids = set(tokenizer.all_special_ids)
geometry_table = {}

for var_name, student in reloaded_students.items():
    student.eval()
    pooled_vectors = []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            # Uncorrupted text forwarded to student encoder
            hidden_states = student(input_ids, attention_mask)
            pooled = masked_mean_pooling(hidden_states, attention_mask, input_ids, special_ids)
            pooled_vectors.append(pooled.cpu())
    
    Z = torch.cat(pooled_vectors, dim=0).numpy()
    geom = compute_representation_geometry(Z)
    geometry_table[var_name] = geom

print(f"{'Variant':<12} | {'RankMe (Singular)':<18} | {'Covariance Rank':<16} | {'Mean Dim Var':<14} | {'Dead Dims %':<12} | {'Cosine Std':<10}")
print("-" * 95)
for var_name, g in geometry_table.items():
    rank_sv = f"{g['effective_rank_singular']:.2f}" if not g['is_degenerate'] else "DEGENERATE"
    rank_cov = f"{g['effective_rank_covariance']:.2f}" if not g['is_degenerate'] else "DEGENERATE"
    mean_var = f"{g['mean_dimension_variance']:.4f}" if not g['is_degenerate'] else "0.0"
    dead_frac = f"{g['dead_dimension_fraction']*100:.1f}%"
    cos_std = f"{g['std_pairwise_cosine']:.4f}" if not g['is_degenerate'] else "N/A"
    print(f"{var_name:<12} | {rank_sv:<18} | {rank_cov:<16} | {mean_var:<14} | {dead_frac:<12} | {cos_std:<10}")

### 12. Downstream Transfer: Frozen Linear Probe on AG News
Fits a `StandardScaler` strictly on training representations, fits logistic regression, and evaluates accuracy and macro-F1 on held-out test representations for all three variants.

In [ ]:
import numpy as np
from datasets import load_dataset
from src.evaluate import evaluate_frozen_linear_probe

ag_train = load_dataset("fancyzhx/ag_news", split="train[:200]")
ag_test = load_dataset("fancyzhx/ag_news", split="test[:100]")

def extract_features(student, dataset):
    features, labels = [], []
    with torch.no_grad():
        for item in dataset:
            enc = tokenizer(item["text"], max_length=128, truncation=True, padding="max_length", return_tensors="pt")
            inp = enc["input_ids"].to(device)
            att = enc["attention_mask"].bool().to(device)
            h = student(inp, att)
            p = masked_mean_pooling(h, att, inp, special_ids)
            features.append(p.squeeze(0).cpu().numpy())
            labels.append(item["label"])
    return np.array(features), np.array(labels)

probe_table = {}
for var_name, student in reloaded_students.items():
    X_train, y_train = extract_features(student, ag_train)
    X_test, y_test = extract_features(student, ag_test)
    res = evaluate_frozen_linear_probe(X_train, y_train, X_test, y_test, seed=42)
    probe_table[var_name] = res

print(f"{'Variant':<12} | {'Test Accuracy':<15} | {'Macro-F1':<12} | {'Majority Baseline':<18}")
print("-" * 65)
for var_name, p in probe_table.items():
    print(f"{var_name:<12} | {p['accuracy']:.4f}          | {p['macro_f1']:.4f}       | {p['majority_baseline']:.4f}")

### 13. Durable Experiment Summary Export
Writes a consolidated JSON evidence manifest to persistent storage documenting hardware, code commit, configurations, geometry metrics, and probe transfer results.

In [ ]:
summary_report = {
    "pinned_commit": PINNED_COMMIT,
    "device": str(device),
    "hardware": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "data_manifest": data_manifest,
    "variants": {
        "mlm_c": {
            "config": config_c.to_dict(),
            "geometry": geometry_table.get("C"),
            "probe": probe_table.get("C"),
        },
        "latent_a": {
            "config": config_a.to_dict(),
            "geometry": geometry_table.get("A"),
            "probe": probe_table.get("A"),
        },
        "latent_b": {
            "config": config_b.to_dict(),
            "geometry": geometry_table.get("B"),
            "probe": probe_table.get("B"),
        },
    },
}

summary_path = DURABLE_ROOT / "three_variant_benchmark_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary_report, f, indent=2)
print(f"Successfully wrote consolidated benchmark summary to {summary_path}")